# 9.1 注意力模式可视化 / Visualizing attention

怎样取出注意力权重并画成热图，怎样据此找出行为特定的注意力头，以及注意力权重为什么不足以单独作为解释。

How attention weights are extracted and drawn as heatmaps, how heads with a specific behaviour are found from them, and why attention weights alone are not sufficient as an explanation.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 提取注意力并绘制热图 / Extract attention and plot a heatmap

从 GPT-2 small 中取出全部注意力权重，计算每个头的平均熵，并把一个头画成热图。测试检查张量的形状、每一行和为 1、因果掩码以及熵在两个手算例子上的值。请查看打印出的观察结果，并打开保存的图片。首次运行会下载约 0.5 GB 的模型。

Extract all attention weights from GPT-2 small, compute the mean entropy of every head and draw one head as a heatmap. The tests check the shape of the tensor, that every row sums to 1, the causal mask, and the entropy on two examples that can be computed by hand. Look at the printed observations and open the saved image. The first run downloads a model of about 0.5 GB.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_gpt2():
    """GPT-2 small. attn_implementation="eager" makes the model return its attention weights."""
    tok = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").eval()
    return model, tok


@torch.no_grad()
def get_attention(model, ids):
    """ids: LongTensor (T,). Returns the attention weights of all heads, shape (n_layer, n_head, T, T).

    Entry [l, h, i, j] is the weight with which head h of layer l, at position i, reads position j.
    """
    # TODO 1: call the model with output_attentions=True. out.attentions is a tuple with one
    #         tensor of shape (1, n_head, T, T) per layer. Stack them into one tensor.
    raise NotImplementedError


def attention_entropy(att):
    """Entropy in nats of every attention row, averaged over the positions. att (L, H, T, T) -> (L, H)."""
    # TODO 2: H = -sum_j a_ij log a_ij. Clamp before the log, masked entries are exactly 0.
    raise NotImplementedError


def plot_head(att, tokens, layer, head, path):
    """Save a heatmap of one head. Rows are the reading positions, columns the positions read."""
    # TODO 3: plt.subplots, ax.imshow(att[layer, head], cmap="Blues", vmin=0, vmax=1),
    #         label both axes with the tokens, fig.savefig(path), plt.close(fig)
    raise NotImplementedError

In [ ]:
# 检查 / checks
import os, tempfile, math
import torch

model, tok = load_gpt2()
text = "The cat sat on the mat because it was tired"
ids = tok(text, return_tensors="pt").input_ids[0]
att = get_attention(model, ids)
T = len(ids)

# 1. Shape: 12 layers, 12 heads, T x T.
assert att.shape == (12, 12, T, T), att.shape

# 2. Every row is a probability distribution.
assert torch.allclose(att.sum(-1), torch.ones(12, 12, T), atol=1e-4), "each row sums to 1"
assert (att >= 0).all()

# 3. The causal mask: no position reads a later one.
upper = torch.triu(torch.ones(T, T), diagonal=1).bool()
assert att[:, :, upper].abs().max() < 1e-6, "weights above the diagonal are 0"
assert torch.allclose(att[:, :, 0, 0], torch.ones(12, 12), atol=1e-5), "the first position can only read itself"

# 4. Entropy. A uniform row over n positions has entropy log n, a one-hot row has entropy 0.
fake = torch.zeros(1, 2, 4, 4)
fake[0, 0] = torch.tril(torch.ones(4, 4)) / torch.arange(1, 5)[:, None]      # uniform over the visible positions
fake[0, 1] = torch.eye(4)                                                    # every position reads itself only
e = attention_entropy(fake)
assert e.shape == (1, 2)
assert abs(e[0, 0].item() - sum(math.log(n) for n in range(1, 5)) / 4) < 1e-5
assert abs(e[0, 1].item()) < 1e-6
ent = attention_entropy(att)
assert ent.shape == (12, 12) and torch.isfinite(ent).all()

# 5. The heatmap is written to disk.
path = os.path.join(tempfile.mkdtemp(), "head.png")
tokens = [tok.decode(i) for i in ids]
plot_head(att, tokens, 4, 11, path)
assert os.path.getsize(path) > 2000

# Observations
sharp = ent.flatten().argmin().item()
flat = ent.flatten().argmax().item()
print(f"tokens: {tokens}")
print(f"sharpest head: layer {sharp // 12} head {sharp % 12}, mean entropy {ent.flatten()[sharp]:.2f} nats")
print(f"most diffuse head: layer {flat // 12} head {flat % 12}, mean entropy {ent.flatten()[flat]:.2f} nats (uniform would be {sum(math.log(n) for n in range(1, T + 1)) / T:.2f})")
print(f"mean weight on the first token over all heads and positions after the first: {att[:, :, 1:, 0].mean():.2f}")
i, c = tokens.index(" it"), tokens.index(" cat")
w = att[:, :, i, c]
for j in w.flatten().topk(3).indices.tolist():
    print(f"layer {j // 12} head {j % 12}, reading from ' it', puts {w.flatten()[j]:.2f} on ' cat'")
print(f"heatmap of layer 4 head 11 saved to {path}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`out.attentions` 是长度为 12 的元组，每个元素的形状是 `(1, 12, T, T)`。`torch.stack([a[0] for a in out.attentions])` 得到 `(12, 12, T, T)`。

`out.attentions` is a tuple of length 12 whose elements have shape `(1, 12, T, T)`. `torch.stack([a[0] for a in out.attentions])` gives `(12, 12, T, T)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

熵的写法是 `-(att * torch.log(att.clamp_min(1e-12))).sum(-1).mean(-1)`。不做 clamp 时，被掩码的项会给出 `0 * log 0 = nan`。

The entropy is `-(att * torch.log(att.clamp_min(1e-12))).sum(-1).mean(-1)`. Without the clamp the masked entries give `0 * log 0 = nan`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_gpt2():
    """GPT-2 small. attn_implementation="eager" makes the model return its attention weights."""
    tok = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").eval()
    return model, tok


@torch.no_grad()
def get_attention(model, ids):
    """ids: LongTensor (T,). Returns the attention weights of all heads, shape (n_layer, n_head, T, T).

    Entry [l, h, i, j] is the weight with which head h of layer l, at position i, reads position j.
    """
    out = model(ids[None], output_attentions=True)
    return torch.stack([a[0] for a in out.attentions])


def attention_entropy(att):
    """Entropy in nats of every attention row, averaged over the positions. att (L, H, T, T) -> (L, H)."""
    ent = -(att * torch.log(att.clamp_min(1e-12))).sum(-1)
    return ent.mean(-1)


def plot_head(att, tokens, layer, head, path):
    """Save a heatmap of one head. Rows are the reading positions, columns the positions read."""
    a = att[layer, head].numpy()
    fig, ax = plt.subplots(figsize=(0.45 * len(tokens) + 1.5, 0.45 * len(tokens) + 1.5))
    im = ax.imshow(a, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens)
    ax.set_xlabel("position read (key)")
    ax.set_ylabel("reading position (query)")
    ax.set_title(f"layer {layer} head {head}")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)
```

</details>

## 2. 找出前一 token 头和归纳头 / Find previous-token heads and induction heads

实现两个打分函数。前一 token 得分是每个位置读取其前一个位置的平均权重。归纳得分在重复两遍的随机序列上计算，是第二遍的每个位置读取“上一次出现的同一 token 的后一个位置”的平均权重。测试先在手工构造的注意力模式上检查两个函数，再在 GPT-2 small 上检查找到的头。

Implement two scoring functions. The previous-token score is the mean weight with which each position reads the position before it. The induction score is computed on a random sequence repeated twice and is the mean weight with which each position of the second copy reads the position after the previous occurrence of the same token. The tests check both functions on hand-made attention patterns and then check the heads found in GPT-2 small.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch


def prev_token_score(att):
    """Mean weight each head puts on the position directly before the reading position.

    att (L, H, T, T) -> (L, H). Position 0 has no previous position and is left out.
    """
    # TODO 1: the entries att[l, h, i, i - 1] for i = 1 .. T-1, averaged over i
    raise NotImplementedError


def repeated_tokens(vocab_size, n, bos_id, seed=0):
    """BOS followed by n random tokens and the same n tokens again. Returns a LongTensor (2n + 1,)."""
    g = torch.Generator().manual_seed(seed)
    # TODO 2: half = torch.randint(0, vocab_size, (n,), generator=g), then concatenate BOS, half, half
    raise NotImplementedError


def induction_score(att, n):
    """Mean weight on the token after the previous occurrence of the current token.

    The sequence is BOS, x_1 .. x_n, x_1 .. x_n. Position i in the second half holds the same token
    as position i - n. An induction head reads from i the position i - n + 1, whose token is the
    one that followed last time. att (L, H, 2n+1, 2n+1) -> (L, H).
    """
    # TODO 3: the entries att[l, h, i, i - n + 1] for i = n+1 .. 2n, averaged over i
    raise NotImplementedError


def top_heads(score, k):
    """The k heads with the largest score as a list of (layer, head), best first."""
    # TODO 4: flatten, topk, convert the flat index back to (layer, head)
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch

# 1. Hand-made patterns.
T = 6
prev = torch.zeros(T, T); prev[0, 0] = 1
for i in range(1, T): prev[i, i - 1] = 1
self_ = torch.eye(T)
fake = torch.stack([prev, self_])[None]                    # (1, 2, T, T)
s = prev_token_score(fake)
assert s.shape == (1, 2)
assert torch.allclose(s, torch.tensor([[1.0, 0.0]]))

# 2. The repeated sequence.
seq = repeated_tokens(50257, 20, 50256, seed=0)
assert seq.shape == (41,) and seq[0] == 50256
assert torch.equal(seq[1:21], seq[21:41])
assert torch.equal(seq, repeated_tokens(50257, 20, 50256, seed=0))
assert not torch.equal(seq, repeated_tokens(50257, 20, 50256, seed=1))

# 3. A hand-made induction pattern for n = 3: positions 4, 5, 6 read positions 2, 3, 4.
n = 3
ind = torch.zeros(7, 7)
for i in range(0, 4): ind[i, 0] = 1
for i in range(4, 7): ind[i, i - n + 1] = 1
copy = torch.zeros(7, 7)
for i in range(0, 4): copy[i, 0] = 1
for i in range(4, 7): copy[i, i - n] = 1                    # reads the earlier copy of the same token instead
s = induction_score(torch.stack([ind, copy])[None], n)
assert torch.allclose(s, torch.tensor([[1.0, 0.0]]))
assert top_heads(torch.tensor([[0.1, 0.9], [0.5, 0.3]]), 2) == [(0, 1), (1, 0)]

# 4. GPT-2 small.
model, tok = load_gpt2()
text = "The cat sat on the mat because it was tired and the dog went to sleep in the garden"
att = get_attention(model, tok(text, return_tensors="pt").input_ids[0])
ps = prev_token_score(att)
best_prev = top_heads(ps, 3)
assert best_prev[0] == (4, 11), best_prev
assert ps[4, 11] > 0.9

scores = torch.stack([induction_score(get_attention(model, repeated_tokens(50257, 20, tok.bos_token_id, seed=s)), 20) for s in range(5)]).mean(0)
best_ind = top_heads(scores, 6)
assert (5, 5) in best_ind[:3] and scores[5, 5] > 0.8, best_ind
assert all(l >= 5 for l, h in best_ind), "induction heads sit above the previous-token head"
assert scores[:4].max() < 0.2, "no induction heads in the first layers"

print("previous-token heads:", ", ".join(f"L{l}H{h} {ps[l, h]:.2f}" for l, h in best_prev))
print("induction heads (mean over 5 random sequences):", ", ".join(f"L{l}H{h} {scores[l, h]:.2f}" for l, h in best_ind))
print(f"heads with induction score above 0.5: {(scores > 0.5).sum().item()} of 144")

# The effect on the prediction: the second half is predictable, the first is not.
ids = repeated_tokens(50257, 20, tok.bos_token_id, seed=0)
with torch.no_grad():
    lp = torch.log_softmax(model(ids[None]).logits[0], -1)
nll = -lp[:-1].gather(1, ids[1:, None])[:, 0]
print(f"loss on the first half {nll[:20].mean():.2f} nats (random guessing would be {torch.log(torch.tensor(50257.0)):.2f}), on the second half {nll[20:].mean():.2f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

用整数索引一次取出所有需要的项。`i = torch.arange(1, T)`，`att[:, :, i, i - 1]` 的形状是 `(L, H, T - 1)`。

Take all required entries at once with integer indexing. With `i = torch.arange(1, T)`, `att[:, :, i, i - 1]` has shape `(L, H, T - 1)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

序列是起始符加两遍 $n$ 个 token。第二遍占据位置 $n+1$ 到 $2n$，位置 $i$ 上的 token 上一次出现在位置 $i - n$，它后面的位置是 $i - n + 1$。

The sequence is the start token and two copies of $n$ tokens. The second copy occupies positions $n+1$ to $2n$, the token at position $i$ last occurred at position $i - n$, and the position after that is $i - n + 1$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


def prev_token_score(att):
    """Mean weight each head puts on the position directly before the reading position.

    att (L, H, T, T) -> (L, H). Position 0 has no previous position and is left out.
    """
    T = att.shape[-1]
    i = torch.arange(1, T)
    return att[:, :, i, i - 1].mean(-1)


def repeated_tokens(vocab_size, n, bos_id, seed=0):
    """BOS followed by n random tokens and the same n tokens again. Returns a LongTensor (2n + 1,)."""
    g = torch.Generator().manual_seed(seed)
    half = torch.randint(0, vocab_size, (n,), generator=g)
    return torch.cat([torch.tensor([bos_id]), half, half])


def induction_score(att, n):
    """Mean weight on the token after the previous occurrence of the current token.

    The sequence is BOS, x_1 .. x_n, x_1 .. x_n. Position i in the second half holds the same token
    as position i - n. An induction head reads from i the position i - n + 1, whose token is the
    one that followed last time. att (L, H, 2n+1, 2n+1) -> (L, H).
    """
    i = torch.arange(n + 1, 2 * n + 1)
    return att[:, :, i, i - n + 1].mean(-1)


def top_heads(score, k):
    """The k heads with the largest score as a list of (layer, head), best first."""
    H = score.shape[1]
    idx = score.flatten().topk(k).indices.tolist()
    return [(j // H, j % H) for j in idx]
```

</details>

## 3. 用消融检验归纳头的作用 / Test the role of induction heads by ablation

用前向预钩子把指定注意力头的输出置零，并在退出时移除钩子。测试检查不消融时输出不变、退出后模型恢复原状、被消融的头对应的切片确实为零，随后比较去掉归纳头与去掉随机选取的头对重复序列损失的影响。请阅读打印出的全部结果，其中包括单个头的消融和普通文本上的结果。运行时间约 20 秒。

Use forward pre-hooks to zero the output of given attention heads and remove the hooks on exit. The tests check that the output is unchanged without ablation, that the model is restored after exit and that the slices of the ablated heads are indeed zero, and then compare the effect on the loss of repeated sequences of removing induction heads and of removing randomly chosen heads. Read all printed results, which include the ablation of single heads and the result on ordinary text. The run takes about 20 seconds.

需要联网下载模型。/ Downloads a model.

In [ ]:
import contextlib
import torch


@contextlib.contextmanager
def ablate_heads(model, heads):
    """Zero the output of the given heads, a list of (layer, head), while the block is active.

    In GPT-2 the outputs of the 12 heads of a layer are concatenated into one vector of size 768,
    head h occupying the slice [h * 64, (h + 1) * 64), and then passed through the linear layer
    attn.c_proj. Zeroing that slice of the input of c_proj removes the head.
    """
    n_head = model.config.n_head
    d_head = model.config.n_embd // n_head
    handles = []
    # TODO 1: group the heads by layer
    # TODO 2: for every layer write a function hook(module, args) that clones args[0], zeroes the
    #         slices of the heads and returns the new input as a tuple (x,)
    # TODO 3: register it with model.transformer.h[l].attn.c_proj.register_forward_pre_hook(hook)
    #         and keep the returned handle in handles
    raise NotImplementedError
    try:
        yield
    finally:
        for hd in handles:
            hd.remove()          # without this the model would stay modified


@torch.no_grad()
def token_losses(model, ids):
    """Cross-entropy in nats of every next-token prediction. ids (T,) -> (T - 1,)."""
    # TODO 4: log_softmax of the logits, then pick the log-probability of the token that follows
    raise NotImplementedError


def second_half_loss(model, ids, n):
    """Mean loss on the second copy of a sequence BOS, x_1 .. x_n, x_1 .. x_n."""
    # TODO 5: predictions 0 .. n-1 are for the first copy, n .. 2n-1 for the second
    raise NotImplementedError

In [ ]:
# 检查 / checks
import random
import torch

model, tok = load_gpt2()
N = 20
seqs = [repeated_tokens(50257, N, tok.bos_token_id, seed=s) for s in range(5)]

# 1. token_losses agrees with the loss computed by the model.
ids = seqs[0]
with torch.no_grad():
    ref = model(ids[None], labels=ids[None]).loss.item()
tl = token_losses(model, ids)
assert tl.shape == (2 * N,)
assert abs(tl.mean().item() - ref) < 1e-4

# 2. Ablating nothing changes nothing, and the hooks are removed afterwards.
with torch.no_grad():
    base_logits = model(ids[None]).logits
    with ablate_heads(model, []):
        assert torch.equal(model(ids[None]).logits, base_logits)
    with ablate_heads(model, [(5, 5)]):
        changed = model(ids[None]).logits
    assert not torch.allclose(changed, base_logits, atol=1e-3), "ablating a head changes the output"
    assert torch.equal(model(ids[None]).logits, base_logits), "the hooks have to be removed on exit"
assert all(len(b.attn.c_proj._forward_pre_hooks) == 0 for b in model.transformer.h)

# 3. Ablating a head equals zeroing its slice by hand.
with torch.no_grad():
    captured = {}
    def grab(module, args): captured["x"] = args[0].clone()
    with ablate_heads(model, [(5, 1), (5, 5)]):
        h0 = model.transformer.h[5].attn.c_proj.register_forward_pre_hook(grab)     # runs after the ablation hook
        model(ids[None])
        h0.remove()
x = captured["x"]
assert x[..., 64:128].abs().max() == 0 and x[..., 320:384].abs().max() == 0, "heads 1 and 5 of layer 5 are zero"
assert x[..., :64].abs().max() > 0 and x[..., 128:320].abs().max() > 0, "the other heads are untouched"

# 4. The causal test. Induction heads are found from attention, their role is tested by removing them.
scores = torch.stack([induction_score(get_attention(model, s), N) for s in seqs]).mean(0)
ind = top_heads(scores, 6)
mean_loss = lambda heads: sum(second_half_loss_with(heads, s) for s in seqs) / len(seqs)
def second_half_loss_with(heads, s):
    with ablate_heads(model, heads):
        return second_half_loss(model, s, N)
base = mean_loss([])
abl3 = mean_loss(ind[:3])
abl6 = mean_loss(ind)
rng = random.Random(0)
pool = [(l, h) for l in range(12) for h in range(12) if (l, h) not in ind]
rand6 = [mean_loss(rng.sample(pool, 6)) for _ in range(10)]
assert abl6 > abl3 > base
assert abl6 > base + 2.0, "removing the induction heads destroys most of the in-context prediction"
assert abl6 > max(rand6), "more than removing any of 10 random sets of 6 other heads"

print(f"second-half loss, mean over {len(seqs)} sequences")
print(f"  intact model                      {base:.2f}")
print(f"  top 3 induction heads removed     {abl3:.2f}")
print(f"  top 6 induction heads removed     {abl6:.2f}   heads: " + " ".join(f"L{l}H{h}" for l, h in ind))
print(f"  6 random other heads removed      {sum(rand6) / len(rand6):.2f} on average, range {min(rand6):.2f} to {max(rand6):.2f} over 10 draws")

# Attention weight and causal effect are not the same thing.
prev = (4, 11)
single = {hd: mean_loss([hd]) - base for hd in ind + [prev]}
for (l, h), d in single.items():
    kind = "previous-token" if (l, h) == prev else "induction"
    print(f"  L{l}H{h} alone ({kind}, score {scores[l, h]:.2f}): loss change {d:+.2f}")

# The same heads on ordinary text.
text = "The Eiffel Tower is in Paris. It was built for the 1889 World's Fair and was the tallest structure in the world until 1930."
tids = tok(text, return_tensors="pt").input_ids[0]
t0 = token_losses(model, tids).mean().item()
with ablate_heads(model, ind):
    t1 = token_losses(model, tids).mean().item()
print(f"ordinary text: loss {t0:.2f} intact, {t1:.2f} with the 6 induction heads removed")

<details><summary><b>提示 1 / Hint 1</b></summary>

预钩子的签名是 `hook(module, args)`，`args[0]` 是 c_proj 的输入，形状为 `(1, T, 768)`。返回 `(x,)` 即可替换输入。应先 `clone()` 再修改。

A pre-hook has the signature `hook(module, args)`, and `args[0]` is the input of c_proj with shape `(1, T, 768)`. Returning `(x,)` replaces the input. Call `clone()` before modifying.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

在循环里定义钩子时，用一个外层函数 `make_hook(hs)` 把当前层的头列表固定下来。直接在循环体中写闭包会使所有钩子都引用最后一层的列表。

When defining hooks in a loop, fix the list of heads of the current layer with an outer function `make_hook(hs)`. A closure written directly in the loop body makes all hooks refer to the list of the last layer.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

`register_forward_pre_hook` 返回一个句柄，`handle.remove()` 移除钩子。把 `yield` 放在 `try` 中、移除放在 `finally` 中，可以保证出错时模型也会恢复。

`register_forward_pre_hook` returns a handle, and `handle.remove()` removes the hook. Putting `yield` inside `try` and the removal inside `finally` restores the model even when an error occurs.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import contextlib
import torch


@contextlib.contextmanager
def ablate_heads(model, heads):
    """Zero the output of the given heads, a list of (layer, head), while the block is active.

    In GPT-2 the outputs of the 12 heads of a layer are concatenated into one vector of size 768,
    head h occupying the slice [h * 64, (h + 1) * 64), and then passed through the linear layer
    attn.c_proj. Zeroing that slice of the input of c_proj removes the head.
    """
    n_head = model.config.n_head
    d_head = model.config.n_embd // n_head
    by_layer = {}
    for l, h in heads:
        by_layer.setdefault(l, []).append(h)

    def make_hook(hs):
        def hook(module, args):
            x = args[0].clone()
            for h in hs:
                x[..., h * d_head:(h + 1) * d_head] = 0
            return (x,)
        return hook

    handles = [model.transformer.h[l].attn.c_proj.register_forward_pre_hook(make_hook(hs)) for l, hs in by_layer.items()]
    try:
        yield
    finally:
        for hd in handles:
            hd.remove()


@torch.no_grad()
def token_losses(model, ids):
    """Cross-entropy in nats of every next-token prediction. ids (T,) -> (T - 1,)."""
    logp = torch.log_softmax(model(ids[None]).logits[0], -1)
    return -logp[:-1].gather(1, ids[1:, None])[:, 0]


def second_half_loss(model, ids, n):
    """Mean loss on the second copy of a sequence BOS, x_1 .. x_n, x_1 .. x_n."""
    return token_losses(model, ids)[n:].mean().item()
```

</details>

## 4. 注意力权重说明了什么，没有说明什么 / What attention weights show and what they do not

在纸上完成。
(a) 位置 $i$ 可以读取 $i$ 个位置（含自身）。说明一行注意力权重的熵最小为 0，最大为 $\log i$，并指出各自在什么情况下取到。
(b) 一个头在位置 3 的权重为 $a = (0.1, 0.8, 0.1)$，三个位置的值向量经过 $W_O W_V$ 之后的范数分别为 $5, 0.2, 3$。计算各位置贡献 $c_{3j}$ 的范数，并说明哪个位置对这个头的输出影响最大。
(c) GPT-2 中一层的 12 个头的输出拼接成 $z = [z_0; \dots; z_{11}]$，再经过线性层 $\text{c\_proj}(z) = W z + b$。说明把 $z_h$ 置零恰好等于从这一层的输出中去掉第 $h$ 个头的那一项，而偏置 $b$ 不受影响。
(d) 假设观察到模型在生成答案时，有若干个头把很高的权重放在工具返回的结果上，模型的回答却与工具结果不一致。给出两种不同的解释，并各设计一个可以区分它们的实验。

Work on paper.
(a) Position $i$ can read $i$ positions (itself included). Show that the entropy of a row of attention weights is at least 0 and at most $\log i$, and say when each is attained.
(b) A head has the weights $a = (0.1, 0.8, 0.1)$ at position 3, and the value vectors of the three positions have norms $5, 0.2, 3$ after $W_O W_V$. Compute the norms of the contributions $c_{3j}$ and say which position affects the output of this head most.
(c) In GPT-2 the outputs of the 12 heads of a layer are concatenated into $z = [z_0; \dots; z_{11}]$ and passed through the linear layer $\text{c\_proj}(z) = W z + b$. Show that zeroing $z_h$ removes exactly the term of head $h$ from the output of the layer, leaving the bias $b$ untouched.
(d) Suppose that while the model generates its answer several heads put a high weight on the result returned by a tool, and the answer nonetheless disagrees with the tool result. Give two different explanations and design for each an experiment that can tell them apart.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 熵是凹函数，在均匀分布处取最大值。

(a) Entropy is concave and is maximal at the uniform distribution.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 把 $W$ 按列分成 12 块，$W z = \sum_h W_h z_h$。

(c) Split $W$ by columns into 12 blocks, so that $W z = \sum_h W_h z_h$.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

(d) 参考推导 1 的第 3 步和第 4 步，以及正文中关于冗余和间接作用的讨论。

(d) See steps 3 and 4 of derivation 1 and the discussion of redundancy and indirect effects in the text.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 熵 $H = -\sum_j a_j \log a_j$ 的每一项都非负，所以 $H \ge 0$，当全部权重集中在一个位置时等号成立。对上界，由 Jensen 不等式，$H = \sum_j a_j \log \frac{1}{a_j} \le \log \sum_j a_j \cdot \frac{1}{a_j} = \log i$，当 $a_j = 1/i$ 即均匀分布时等号成立。第一个位置只能读取自身，熵恒为 0。

(b) $\lVert c_{31} \rVert = 0.1 \times 5 = 0.5$，$\lVert c_{32} \rVert = 0.8 \times 0.2 = 0.16$，$\lVert c_{33} \rVert = 0.1 \times 3 = 0.3$。权重最高的位置 2 贡献最小，权重只有 0.1 的位置 1 贡献最大。这里只比较了范数，方向是否与输出相关还需要另行考察。

(c) 把 $W$ 按列分成 12 块 $W = [W_0, \dots, W_{11}]$，每块 64 列，则 $W z + b = \sum_{h} W_h z_h + b$。把 $z_h$ 置零后，和中恰好少了 $W_h z_h$ 这一项，其他头的项和偏置 $b$ 都不变。这就是练习 3 在 c_proj 的输入上操作的原因。如果改为在 c_proj 的输出上操作，各个头的贡献已经加在一起，无法单独去掉其中一个。

(d) 解释一，这些头读取了工具结果，但写入残差流的内容很小，或者方向与答案 token 无关。它们可能只是在读取格式信息，例如括号和引号的位置。检验办法是计算这些头在答案位置上的输出对“工具给出的答案”与“模型给出的答案”两者 logit 差的直接贡献（9.2 的方法），并消融这些头看回答是否改变。

解释二，这些头确实把工具给出的答案写向了输出，但别的组件以更大的幅度写入了模型自身记忆中的答案，例如靠后的前馈子层。检验办法是用 logit lens 逐层查看两个候选答案的名次，看工具给出的答案是否在某些层占上风而后被压过，再对起压制作用的组件做消融或修补（9.3）。

两种解释并不互斥。只凭注意力权重无法区分它们，这正是需要干预实验的原因。

---

(a) Every term of the entropy $H = -\sum_j a_j \log a_j$ is non-negative, so $H \ge 0$, with equality when all weight sits on one position. For the upper bound, by the inequality of Jensen, $H = \sum_j a_j \log \frac{1}{a_j} \le \log \sum_j a_j \cdot \frac{1}{a_j} = \log i$, with equality for $a_j = 1/i$, the uniform distribution. The first position can read only itself and its entropy is always 0.

(b) $\lVert c_{31} \rVert = 0.1 \times 5 = 0.5$, $\lVert c_{32} \rVert = 0.8 \times 0.2 = 0.16$, $\lVert c_{33} \rVert = 0.1 \times 3 = 0.3$. Position 2 with the highest weight contributes least, and position 1 with a weight of only 0.1 contributes most. Only norms are compared here, and whether the directions are relevant to the output has to be examined separately.

(c) Split $W$ by columns into 12 blocks $W = [W_0, \dots, W_{11}]$ of 64 columns each. Then $W z + b = \sum_{h} W_h z_h + b$. With $z_h$ set to zero the sum lacks exactly the term $W_h z_h$, and the terms of the other heads and the bias $b$ are unchanged. This is why exercise 3 operates on the input of c_proj. On the output of c_proj the contributions of the heads are already added together and a single one can no longer be removed.

(d) Explanation one. The heads read the tool result, while what they write into the residual stream is small or points in a direction unrelated to the answer tokens. They may be reading formatting information alone, such as the positions of brackets and quotation marks. The test is to compute the direct contribution of the outputs of these heads at the answer position to the logit difference between the answer given by the tool and the answer given by the model (the method of 9.2), and to ablate the heads and see whether the answer changes.

Explanation two. The heads do write the answer given by the tool toward the output, while other components write the answer from the memory of the model with a larger magnitude, for instance late feed-forward sublayers. The test is to follow the ranks of the two candidate answers layer by layer with the logit lens, to see whether the answer of the tool leads in some layers and is then overtaken, and to ablate or patch the components that do the overriding (9.3).

The two explanations do not exclude each other. Attention weights alone cannot tell them apart, which is the reason intervention experiments are needed.

</details>